In [12]:
import os, sys, json, time, random, re, io
from typing import List, Dict, Tuple
import requests
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, ConfusionMatrixDisplay, confusion_matrix, classification_report
import matplotlib.pyplot as plt

# =========================
# تنظیمات اصلی
# =========================
AVALAI_BASE_URL = "https://api.avalai.ir/v1"   # یا "https://api.avalapis.ir/v1"
AVALAI_API_KEY = "aa-hXsC7ulscaBbPcU6663EvyHVCiyty0HKu2ar4UUXCVU0W89Y"   # حتماً ست شود
MODEL_NAME = "gpt-4o"
SAMPLE_SIZE = 1        # «سمپل کوچک»؛ می‌توانید کمتر/بیشتر کنید
SEED = 42               # برای تکرارپذیری
TIMEOUT = 60            # ثانیه
MAX_RETRIES = 3

# آدرس‌های خام گیت‌هاب برای اسپلیت‌ها
DATA_URLS = [
    "https://raw.githubusercontent.com/MichSchli/AVeriTeC/main/data/test.json",
    "https://raw.githubusercontent.com/MichSchli/AVeriTeC/main/data/dev.json",
]

# لیبل‌های مجاز مطابق توضیحات دیتاست (حساس به حروف نیست؛ نرمالایز می‌کنیم)
CANON_LABELS = [
    "SUPPORTED",
    "REFUTED",
    "NOT ENOUGH EVIDENCE",
    "CONFLICTING EVIDENCE/CHERRY-PICKING",
]

# =========================
# ابزارک‌ها
# =========================
def ensure_api_key():
    if not AVALAI_API_KEY:
        raise RuntimeError(
            "AVALAI_API_KEY در محیط ست نشده. قبل از اجرا، کلید را به‌صورت متغیر محیطی تنظیم کنید."
        )

def fetch_json_from_urls(urls: List[str]) -> List[Dict]:
    last_err = None
    for url in urls:
        try:
            r = requests.get(url, timeout=60)
            r.raise_for_status()
            data = r.json()
            if isinstance(data, list):
                return data
            # برخی ریپوها ممکن است dict با کلیدها داشته باشند؛ اینجا فقط لیست ادعاها را می‌خواهیم
            if isinstance(data, dict) and "data" in data and isinstance(data["data"], list):
                return data["data"]
            raise ValueError("ساختار JSON غیرمنتظره است.")
        except Exception as e:
            last_err = e
            continue
    raise RuntimeError(f"دانلود/خواندن دیتاست ناموفق بود. آخرین خطا: {last_err}")

def sample_dataset(records: List[Dict], n: int, seed: int) -> List[Dict]:
    rnd = random.Random(seed)
    if n >= len(records):
        return records
    return rnd.sample(records, n)

def norm_label(text: str) -> str:
    if not text:
        return ""
    t = text.strip().lower()
    # نگاشت‌های احتمالی به فرم استاندارد
    mapping = {
        "supported": "SUPPORTED",
        "refuted": "REFUTED",
        "not enough evidence": "NOT ENOUGH EVIDENCE",
        "nei": "NOT ENOUGH EVIDENCE",
        "conflicting evidence": "CONFLICTING EVIDENCE/CHERRY-PICKING",
        "cherry-picking": "CONFLICTING EVIDENCE/CHERRY-PICKING",
        "conflicting evidence/cherry-picking": "CONFLICTING EVIDENCE/CHERRY-PICKING",
    }
    # نزدیک‌ترین مچ
    for k, v in mapping.items():
        if t == k or k in t:
            return v
    # اگر کاربر عین استاندارد را برگرداند
    for canon in CANON_LABELS:
        if t == canon.lower():
            return canon
    return ""  # نامعتبر

def build_prompt(claim: str, claim_date: str = None) -> List[Dict]:
    labels_bullet = "\n".join(f"- {l}" for l in CANON_LABELS)
    sys_msg = (
        "You are a careful fact-checking assistant. "
        "Given a claim, choose exactly one veracity label from the allowed set. "
        "IMPORTANT: Output ONLY valid JSON like {\"label\": \"<ONE OF THE ALLOWED LABELS>\"} without any explanation."
    )
    date_line = f"\nClaim date (if known): {claim_date}" if claim_date else ""
    user_msg = (
        f"Claim: {claim}{date_line}\n\n"
        "Allowed labels:\n"
        f"{labels_bullet}\n\n"
        "Return JSON only, with a single key 'label'."
    )
    return [
        {"role": "system", "content": sys_msg},
        {"role": "user", "content": user_msg},
    ]

# =========================
# تماس با AvalAI (OpenAI-compatible)
# =========================
def chat_completion(messages: List[Dict], temperature: float = 0.0) -> str:
    """
    تماس ساده با endpoint /v1/chat/completions روی AvalAI.
    خروجی: متن پاسخ مدل (رشته).
    """
    url = f"{AVALAI_BASE_URL}/chat/completions"
    headers = {
        "Authorization": f"Bearer {AVALAI_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": temperature,
    }
    print(messages)

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(url, headers=headers, data=json.dumps(payload), timeout=TIMEOUT)
            print(resp)
            if resp.status_code == 429 or 500 <= resp.status_code < 600:
                # بک‌آف نمایی
                wait = 2 ** (attempt - 1)
                time.sleep(wait)
                continue
            resp.raise_for_status()
            data = resp.json()
            print(data)
            return data["choices"][0]["message"]["content"]
        except Exception as e:
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 ** attempt)
    # در عمل به اینجا نمی‌رسد
    return ""

def extract_label_from_response(text: str) -> str:
    """
    تلاش برای پارس JSON. اگر JSON نبود، با رگ‌اکس برچسب را حدس می‌زند.
    """
    print(text)
    if not text:
        return ""
    # تلاش برای JSON خالص
    try:
        j = json.loads(text)
        cand = j.get("label", "")
        return norm_label(cand)
    except Exception:
        pass
    # رگ‌اکس: "label": "SOMETHING"
    m = re.search(r'"label"\s*:\s*"([^"]+)"', text, flags=re.IGNORECASE)
    if m:
        return norm_label(m.group(1))
    # آخرین تلاش: هرکدام از برچسب‌ها را اگر در متن بود!
    for c in CANON_LABELS:
        if c.lower() in text.lower():
            return c
    return ""

# =========================
# ارزیابی و ترسیم نمودار
# =========================
def evaluate_and_plot(df: pd.DataFrame, out_prefix: str = "averitec_demo"):
    y_true = df["label_true"].tolist()
    y_pred = df["label_pred"].tolist()

    # حذف نمونه‌های پیش‌بینی‌نشده
    keep = [i for i, p in enumerate(y_pred) if p in CANON_LABELS]
    if len(keep) < len(y_pred):
        df = df.iloc[keep]
        y_true = df["label_true"].tolist()
        y_pred = df["label_pred"].tolist()

    acc = accuracy_score(y_true, y_pred) if y_pred else 0.0
    f1 = f1_score(y_true, y_pred, average="macro", labels=CANON_LABELS) if y_pred else 0.0

    print(f"\nAccuracy: {acc:.3f} | Macro-F1: {f1:.3f}\n")
    print(classification_report(y_true, y_pred, labels=CANON_LABELS, zero_division=0))

    # نمودار توزیع برچسب‌های درست/پیش‌بینی
    counts_true = pd.Series(y_true).value_counts().reindex(CANON_LABELS, fill_value=0)
    counts_pred = pd.Series(y_pred).value_counts().reindex(CANON_LABELS, fill_value=0)

    plt.figure(figsize=(8, 4.5))
    plt.title("Label counts (True vs Predicted)")
    x = range(len(CANON_LABELS))
    plt.bar([i - 0.2 for i in x], counts_true.values, width=0.4, label="True")
    plt.bar([i + 0.2 for i in x], counts_pred.values, width=0.4, label="Pred")
    plt.xticks(x, CANON_LABELS, rotation=20, ha="right")
    plt.tight_layout()
    plt.legend()
    plt.savefig(f"{out_prefix}_label_cunts.png", dpi=200)

    # ماتریس اغتشاش
    cm = confusion_matrix(y_true, y_pred, labels=CANON_LABELS, normalize=None)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CANON_LABELS)
    fig, ax = plt.subplots(figsize=(6.5, 6))
    disp.plot(ax=ax, cmap="Blues", xticks_rotation=20, colorbar=False)
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig(f"{out_prefix}_confusion_matrix.png", dpi=220)
    plt.close('all')

    # ذخیره نتایج ردیفی
    df.to_csv(f"{out_prefix}_rows.csv", index=False, encoding="utf-8-sig")

# =========================
# اجرای اصلی
# =========================
def main():
    ensure_api_key()

    # 1) خواندن دیتاست
    print("Downloading AVeriTeC split (test/dev) from GitHub ...")
    records = fetch_json_from_urls(DATA_URLS)
    assert isinstance(records, list) and len(records) > 0, "لیست ادعاها خالی است."

    # هر رکورد شبیه نمونه زیر است (طبق README ریپو):
    # { "claim": "...", "label": "...", "claim_date": "...", "questions": [ ... ], ... }
    # ما فقط claim و label و claim_date را استفاده می‌کنیم.

    # 2) سمپل کوچک
    sample = sample_dataset(records, n=SAMPLE_SIZE, seed=SEED)

    # 3) کوئری مدل روی هر ادعا
    rows = []
    print(f"Evaluating {len(sample)} samples via AvalAI / {MODEL_NAME} ...")
    for rec in tqdm(sample):
        claim = rec.get("claim", "").strip()
        gold = rec.get("label", "").strip()
        claim_date = rec.get("claim_date", "")

        msgs = build_prompt(claim=claim, claim_date=claim_date)
        try:
            raw = chat_completion(messages=msgs, temperature=0.0)
            pred = extract_label_from_response(raw)
            print(pred)
        except Exception as e:
            raw = f"ERROR: {e}"
            pred = ""

        rows.append({
            "claim": claim,
            "label_true": norm_label(gold),
            "label_pred": pred,
            "model_raw": raw,
        })

    df = pd.DataFrame(rows)
    # 4) ارزیابی و نمودار
    evaluate_and_plot(df, out_prefix="averitec_avvalai_gpt35")

if __name__ == "__main__":
    try:
        main()
    except Exception as ex:
        print("FATAL:", ex)
        sys.exit(1)


Evaluating 1 samples via AvalAI / gpt-4o ...


  0%|          | 0/1 [00:00<?, ?it/s]

[{'role': 'system', 'content': 'You are a careful fact-checking assistant. Given a claim, choose exactly one veracity label from the allowed set. IMPORTANT: Output ONLY valid JSON like {"label": "<ONE OF THE ALLOWED LABELS>"} without any explanation.'}, {'role': 'user', 'content': "Claim: Carlos Gimenez approved a 67% pay raise for himself and increased his own pension.\nClaim date (if known): 17-9-2020\n\nAllowed labels:\n- SUPPORTED\n- REFUTED\n- NOT ENOUGH EVIDENCE\n- CONFLICTING EVIDENCE/CHERRY-PICKING\n\nReturn JSON only, with a single key 'label'."}]


100%|██████████| 1/1 [00:00<00:00,  1.04it/s]

<Response [200]>
{'id': 'chatcmpl-CJF69fMKsskOfW6cPWquPDIIgT3E6', 'created': 1758702305, 'model': 'gpt-4o-2024-11-20', 'object': 'chat.completion', 'system_fingerprint': 'fp_ee1d74bde0', 'choices': [{'finish_reason': 'stop', 'index': 0, 'message': {'content': '{"label": "REFUTED"}', 'role': 'assistant', 'annotations': []}}], 'usage': {'completion_tokens': 9, 'prompt_tokens': 138, 'total_tokens': 147, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'text_tokens': None, 'image_tokens': None}}, 'estimated_cost': {'unit': '0.0004785000', 'irt': 51.32, 'exchange_rate': 107250}}
{"label": "REFUTED"}
REFUTED

Accuracy: 0.000 | Macro-F1: 0.000

                                     precision    recall  f1-score   support

                          SUPPORTED       0.00      0.00      0.00       1.0
                       


c:\Users\a\anaconda3\2\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
